In [ ]:
# manejo de datos
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# api Xenocanto
from xenopy import Query
from scipy import signal
import scipy.signal.windows

# manejo de audio
import librosa
import librosa.display

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
def cargar_audios(ruta_base):
    audios_cargados = {}
    ruta_especie = os.path.join(ruta_base)
    if os.path.isdir(ruta_especie):
        especie = os.path.basename(ruta_especie)
        for nombre_comun in os.listdir(ruta_especie):
            ruta_nombre_comun = os.path.join(ruta_especie, nombre_comun)
            if os.path.isdir(ruta_nombre_comun):
                for archivo in os.listdir(ruta_nombre_comun):
                    if archivo.endswith('.mp3'):
                        ruta_archivo = os.path.join(ruta_nombre_comun, archivo)
                        try:
                            audio, sr = librosa.load(ruta_archivo)
                            # Construir el nombre clave como "especie_id.mp3"
                            nombre_clave = f"{especie}_{archivo.split('.')[0]}"
                            audios_cargados[nombre_clave] = (audio, sr)
                        except Exception as e:
                            print(f"Error al cargar {ruta_archivo}: {e}")
    return audios_cargados, especie

In [ ]:
def feature_extraction(audios_cargados, especie, familia):
    all_features = [] 
    keys = list(audios_cargados.keys())
    for key in keys: 
        y, sr = audios_cargados[key]
        features = { 
            'Familia': familia,
            'Genero': especie.split(' ')[0],
            'Especie': especie,
            'id': key,
            'Longitud de onda': len(y),
            'Intensidad miníma': np.min(librosa.feature.rms(y=y)),
            'Intensidad media': np.mean(librosa.feature.rms(y=y)),
            'Intensidad máxima': np.max(librosa.feature.rms(y=y)),
            'Tonos principales': len(librosa.core.piptrack(y=y)[1]),
            'Melodía': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
            'Centroide espectral': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'Rolloff espectral': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
            'Contraste espectral': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
            'Ancho de banda espectral': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            'Croma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
            'Tempo': librosa.beat.tempo(y=y, sr=sr)[0],
            'Pulso': np.mean(librosa.feature.tempogram(y=y, sr=sr)),
            'Coeficientes MFCC': np.mean(librosa.feature.mfcc(y=y, sr=sr)),
            'RMS': np.mean(librosa.feature.rms(y=y)),
            'Cens': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
            'Piptrack': np.mean(librosa.core.piptrack(y=y)[1]),
            'Cruces por cero': np.mean(librosa.feature.zero_crossing_rate(y)),
            'Cromagrama CQT': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)),
            'Cromagrama CENS': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
            'Melspectrogram': np.mean(librosa.feature.melspectrogram(y=y, sr=sr)),
            'Polynomial coefficients': np.mean(librosa.feature.poly_features(y=y, sr=sr)),
            'Fourier tempogram': np.mean(librosa.feature.fourier_tempogram(y=y, sr=sr)),
            'Ceros Cruzados': np.mean(librosa.feature.zero_crossing_rate(y)),
            'Perceptrual Sharpness': np.mean(librosa.feature.spectral_flatness(y=y)),
            'Loudness': np.sum(librosa.feature.rms(y=y)),
            'Tonnetz': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
            'Contraste': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
            'Rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
            'Centroide': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'Bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        }
        all_features.append(features) 
    df = pd.DataFrame(all_features)
    
    return df 

In [ ]:
import os

# Listar todas las familias una sola vez
familias = os.listdir('songs/')
for familia in familias:
    print(f'Procesando familia: {familia}')
    
    # Listar todos los géneros dentro de la familia actual
    generos_path = f'songs/{familia}/'
    generos = os.listdir(generos_path)
    for genero in generos:
        print(f'  Procesando género: {genero}')
        
        # Listar todas las especies dentro del género actual
        especies_path = f'{generos_path}{genero}/'
        especies = os.listdir(especies_path)
        for especie in especies:
            print(f'    Procesando especie: {especie}')
            
            # Construir la ruta base para la especie actual
            ruta_base = f'{especies_path}{especie}/'
            audios_cargados, especie_actualizada = cargar_audios(ruta_base)
            print(f"Se cargaron {len(audios_cargados)} archivos de audio.")
            
            # Verificar si se encontraron audios
            if not audios_cargados:
                print(f'No se encontraron cantos para la especie {especie} en {genero}, {familia}')
                continue  # Salta a la próxima especie
            
            # Extraer características y guardar en CSV
            df = feature_extraction(audios_cargados, especie, familia)
            archivo_csv = f'features/{familia}_{genero}_{especie}.csv'
            df.to_csv(archivo_csv, index=False)
            print(f'Archivo {archivo_csv} guardado exitosamente.\n')

In [ ]:
## Leer los archivos CSV y concatenar en un solo DataFrame 
ruta_features = 'features/'
archivos_csv = os.listdir(ruta_features)
dfs = []
for archivo_csv in archivos_csv:
    df = pd.read_csv(os.path.join(ruta_features, archivo_csv))
    dfs.append(df)
df_final = pd.concat(dfs, ignore_index=True)
df_final.to_csv('data/features_total.csv', index=False)

In [ ]:
# features csv manualmente

familia = os.listdir('songs/')[4]

try: 
    genero = os.listdir(f'songs/{familia}')[0]
except:
    print(f'No hay mas generos para la familia {familia}')

try:
    especie = os.listdir(f'songs/{familia}/{genero}')[0]
except:
    print(f'No hay mas especies para el genero {genero}')

print(f'familia: {familia}')
print(f'genero: {genero}')
print(f'especie: {especie}')

In [ ]:
ruta_base = f'songs/{familia}/{genero}/{especie}'
audios_cargados, especie = cargar_audios(ruta_base)
print(f"Se cargaron {len(audios_cargados)} archivos de audio.")

In [ ]:
df = feature_extraction(audios_cargados, especie, familia)
print(df.shape)
df.head()

In [ ]:
df.to_csv(f'data/{especie}.csv', index=False)